# 🧭 Notebook 3: BFF vs. API Gateway, pitfalls, and best practices

By now you can build a BFF. This notebook focuses on the **design questions** that come up in practice:

1. When is a BFF the right tool, and when is a plain **API Gateway** enough?
2. What are the **classic pitfalls** (and how to avoid them)?
3. Where should **auth, rate-limiting, and shared logic** live?
4. When should you **not** use a BFF?

## 🛠️ Setup

```bash
cd 05-microservices/bff
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab uses **only the Python standard library** — no servers to start, no Docker. We simulate HTTP calls with plain Python functions so you can focus on the pattern.

## 1. BFF vs. API Gateway — choosing between them

Both sit between clients and microservices, but they serve different purposes.

### API Gateway
A **single shared entry point** for many clients. It handles cross-cutting concerns that *every* client needs the same way:

- TLS termination
- Authentication / token validation
- Rate limiting, quotas
- Request routing to internal services
- Logging, metrics, tracing

It is **client-agnostic** — it doesn't know (or care) that one caller is a phone and another is a TV.

### BFF
A **client-specific** gateway that handles things only *that* client cares about:

- Aggregating several downstream calls into one response
- Reshaping / filtering data for the client's UI
- Client-specific caching, fallbacks, and pre-rendering
- Different auth flows (e.g. mobile tokens vs. session cookies)

### The common production layout

```
clients --> API Gateway --> BFF (web)    --> microservices
                       \-> BFF (mobile) --> microservices
                       \-> BFF (TV)     --> microservices
```

The Gateway does **generic** work once; each BFF does **client-specific** work. The Gateway is owned by a platform team; each BFF is owned by the relevant frontend team.

In [ ]:
# A tiny simulation showing the layering. No new deps - just functions.

def api_gateway(request, downstream_bff):
    """Generic cross-cutting concerns - applies to every client."""
    # 1. Auth check
    if not request.get('token'):
        return {'status': 401, 'body': 'missing token'}
    # 2. Rate-limit (pretend)
    if request.get('rl_exceeded'):
        return {'status': 429, 'body': 'slow down'}
    # 3. Route to the right BFF
    return downstream_bff(request)

def svc_user(uid):   return {'id': uid, 'name': 'Ada', 'email': 'a@x.io'}
def svc_orders(uid): return [{'id': i, 'total': 10*i} for i in range(3)]

def mobile_bff(request):
    uid = request['uid']
    return {'status': 200, 'body': {
        'name': svc_user(uid)['name'],
        'order_count': len(svc_orders(uid)),
    }}

def web_bff(request):
    uid = request['uid']
    return {'status': 200, 'body': {
        'user': svc_user(uid),
        'orders': svc_orders(uid),
    }}

print('unauthenticated  ->', api_gateway({}, mobile_bff))
print('mobile happy path->', api_gateway({'token': 't', 'uid': 1}, mobile_bff))
print('web happy path   ->', api_gateway({'token': 't', 'uid': 1}, web_bff))


## 2. Pitfall #1 — Duplicating business logic across BFFs

When the mobile BFF and the web BFF both need to compute "is this user a VIP?", junior teams often copy-paste the logic into both. Six months later the rules drift apart.

### ⚠️ Bad: copy-pasted business rule

In [ ]:
def is_vip_mobile(orders):
    # mobile team's version
    return sum(o['total'] for o in orders) > 100

def is_vip_web(orders):
    # web team's version - got edited in a rush, now slightly different
    return sum(o['total'] for o in orders) >= 100

sample = [{'total': 100}]
print('mobile thinks VIP?', is_vip_mobile(sample))
print('web    thinks VIP?', is_vip_web(sample))   # different answer!


### ✅ Good: push shared **business** rules into a downstream service (or shared library)

BFFs should contain **presentation** logic (shaping for a UI), not **business** logic (who qualifies as a VIP).

The rule "sum(orders) > 100 → VIP" belongs in a `user-service` or `loyalty-service` endpoint that *every* BFF calls. That way there's one source of truth.

In [ ]:
# Single source of truth - one service, called by every BFF.
def svc_user_status(uid, orders):
    # In real life this lives in a user/loyalty microservice.
    return {'vip': sum(o['total'] for o in orders) > 100}

def mobile_bff_v2(uid):
    orders = svc_orders(uid)
    return {'order_count': len(orders), **svc_user_status(uid, orders)}

def web_bff_v2(uid):
    orders = svc_orders(uid)
    return {'orders': orders, **svc_user_status(uid, orders)}

print('mobile:', mobile_bff_v2(1))
print('web:   ', web_bff_v2(1))


## 3. Pitfall #2 — Letting BFFs call each other

It's tempting for the web BFF to call the mobile BFF "because it already has the logic I need." Don't.

- BFFs are **owned by different teams**; coupling them re-creates the bottleneck BFFs were supposed to remove.
- Each BFF's response shape is optimized for **its** client — consuming it from another BFF is fragile.
- The underlying data lives in **microservices**. Both BFFs should call the service directly.

✅ **Rule:** BFFs call **services**, never other BFFs.

## 4. Pitfall #3 — Putting auth in every BFF

If every BFF re-implements login, token validation, and rate-limiting, you end up with:

- Five different auth bugs instead of one.
- Five places to rotate keys.
- Inconsistent error messages for the same failure.

✅ **Rule:** put **generic security** at the API Gateway; put **client-specific** auth (e.g. mobile push-token exchange) in the relevant BFF only.

## 5. Pitfall #4 — BFF bloat ("every page a new BFF")

Some teams grow a BFF per *screen* instead of per *client type*. You then have 40 BFFs for one app, and they all drift.

### ✅ A good rule of thumb

- **One BFF per client type** (mobile, web, TV, partner API, …).
- Inside a BFF, use **modules / routes** for specific screens — not new services.
- Split a BFF only when **ownership** or **release cadence** truly diverges.

## 6. Pitfall #5 — Invisible BFFs (no correlation ID, no tracing)

A BFF turns **one** client request into **many** downstream calls. When something goes wrong, you need to know *which* phone request caused *which* slow service call. Without a shared identifier you're debugging blind.

### ✅ Good: pass a **correlation ID** through every call

The BFF generates (or forwards) a unique ID, logs it, and passes it to every downstream service. Your log aggregator can then stitch the whole request together.


In [ ]:
import uuid, logging, time
from concurrent.futures import ThreadPoolExecutor

logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger('bff')

def svc(name, cid):
    # Every service logs with the same correlation id.
    log.info(f'  [cid={cid}] {name} service handling request')
    time.sleep(0.01)
    return {'ok': True, 'from': name}

def mobile_bff_traced(request):
    # Use the client's correlation id if it sent one, else make a new one.
    cid = request.get('correlation_id') or uuid.uuid4().hex[:8]
    log.info(f'[cid={cid}] BFF received mobile request')

    with ThreadPoolExecutor(max_workers=2) as ex:
        f_user   = ex.submit(svc, 'user',   cid)
        f_orders = ex.submit(svc, 'orders', cid)
        user, orders = f_user.result(), f_orders.result()

    log.info(f'[cid={cid}] BFF responding to mobile request')
    return {'correlation_id': cid, 'user': user, 'orders': orders}

# Simulate two independent client requests - each has its own trace.
mobile_bff_traced({})
print('---')
mobile_bff_traced({'correlation_id': 'abcd1234'})


## 7. Alternative to a BFF — **GraphQL**

If your main reason for wanting a BFF is "each client wants a different shape of the same data", a **GraphQL** endpoint can solve that without spinning up several gateways. Each client writes its own query and gets exactly the fields it asked for.

| You probably want a **BFF** when…            | You probably want **GraphQL** when…        |
|----------------------------------------------|--------------------------------------------|
| Clients need different *logic* (resilience, pre-rendering, aggregation) | Clients need different *shapes of the same data* |
| You want per-client teams with full ownership | One team can own the schema                |
| Response shaping is complex or stateful      | Shaping is mostly picking fields           |
| You already have an API Gateway in place     | You're starting from scratch               |

Some teams combine the two: a BFF internally uses GraphQL to query microservices. The BFF still exists to hold client-specific *logic*, caching, and fallbacks.


## 8. When **not** to use a BFF

BFFs are not free. Don't reach for one if:

| Situation                                               | Better choice                    |
|---------------------------------------------------------|----------------------------------|
| Only one client, no divergence in sight                 | Plain API (no gateway)           |
| All clients want roughly the same data                  | Single shared API + GraphQL      |
| Tiny team, no ops capacity for extra services           | Start without BFFs, add later    |
| You mostly need auth/routing/rate-limit                 | API Gateway only                 |
| Downstream is already a well-shaped aggregate           | Call it directly from the client |

> If you build a BFF, you take on another deployable, another pager, another CI/CD pipeline. Make sure the **payoff** (shaped payloads, client-team autonomy) is worth it.

## 9. Checklist for a healthy BFF

Use this when reviewing a BFF PR:

- [ ] Owns **only presentation** logic; business rules live downstream.
- [ ] Fans out to downstream services **in parallel** where possible.
- [ ] Handles partial failures with **graceful fallbacks** (non-critical data optional).
- [ ] Times out downstream calls; surfaces `_degraded` to the UI when relevant.
- [ ] Does **not** call other BFFs.
- [ ] Sensitive fields (emails, addresses, security data) are filtered before they leave the BFF.
- [ ] Caches expensive/stable data with a **short TTL**.
- [ ] Observed: has request-level metrics/tracing so you can see per-endpoint latency.
- [ ] Owned by the team that owns the frontend it serves.

## 🧠 Final takeaways

- **BFF = one gateway per client type**, owned by that client's team.
- Put **generic** concerns (auth, rate-limit, TLS) in a shared **API Gateway**; put **client-specific** concerns in the **BFF**.
- Shared **business** rules belong in downstream services — not duplicated across BFFs.
- A BFF is a natural place for **parallel fan-out**, **graceful degradation**, and **small caches**.
- Don't over-split (one BFF per screen) and don't under-split (one BFF for all clients). Split by **client type**.

### 📚 Further reading

- Sam Newman — *"Pattern: Backends For Frontends"* (2015, the canonical write-up).
- Netflix Tech Blog — posts about their device-specific APIs (precursor to BFFs).
- `references/designgurus.md` in this lab — scraped lesson notes that feed this lab.